# 15. Eksperimen Utama: Skenario A dan B, Model Tunggal dan Hybrid (Satu Protokol)

Notebook ini membandingkan **semua konfigurasi pada protokol yang sama** sehingga klaim pada judul penelitian bisa diuji:

| Skenario | Isi |
|---|---|
| **A** | tanpa cost-sensitive learning (hiperparameter dari notebook 15a) |
| **B** | dengan cost-sensitive learning (`class_weight` / `sample_weight` balanced, hiperparameter dari notebook 07 dan 08) |

Untuk tiap skenario dievaluasi empat model: **RF**, **XGBoost**, **Hybrid (meta-learner tanpa bobot)** sesuai proposal, dan **Hybrid (meta-learner berbobot)** sesuai kode yang sudah dibangun (`class_weight="balanced"` pada Logistic Regression).

**Protokol:** Stratified K-Fold luar (`N_OUTER`, bawaan 5). Pada setiap lipatan luar, model dasar dilatih pada data latih luar dan dinilai pada lipatan luar. Untuk Hybrid, probabilitas OOF model dasar dibuat dengan Stratified K-Fold dalam (`N_INNER`) hanya dari data latih luar, lalu meta-learner dilatih dari OOF itu. Meta-learner tidak pernah melihat lipatan luar yang dinilai.

**Keluaran** (folder `hasil_eksperimen/15_eksperimen_utama`):
- `metrik_per_fold.csv`: seluruh metrik per lipatan
- `ringkasan_mean_sd.csv`: rata-rata dan simpangan baku
- `uji_berpasangan.csv`: selisih berpasangan dengan uji t terkoreksi dan Wilcoxon
- `threshold_oof_gabungan.csv`: hasil setelah threshold prediabetes dioptimalkan
- `sensitivitas_kembaran.csv`: hasil dengan dan tanpa baris yang punya kembaran persis di data latih
- `grafik_perbandingan.png`
- folder `ckpt`: hasil mentah per lipatan (probabilitas), dipakai ulang bila notebook dilanjutkan

**Keamanan:**
- Hanya membaca `data/final_split/train_final.csv`. `test_final_locked.csv` tidak dibaca.
- Tidak menulis ke `artifacts` dan `data`.
- Bisa dihentikan dan dilanjutkan: lipatan yang sudah selesai tersimpan di `ckpt` dan tidak dihitung ulang.

**Waktu:** berat. Per pengulangan CV ada `N_OUTER x 2 skenario x (N_INNER + 1)` pelatihan Random Forest dan XGBoost. Perkiraan kasar beberapa jam per pengulangan, jadi jalankan semalam. Gunakan `UJI_CEPAT = True` lebih dulu untuk memastikan semuanya berjalan (1-3 menit, hasilnya ke folder terpisah dan bukan untuk penelitian).

**Catatan metodologi:** hiperparameter diambil dari tuning pada seluruh data latih, jadi estimasi CV ini sedikit optimistis dan sama-sama berlaku untuk semua konfigurasi. Ini keterbatasan yang perlu ditulis di laporan.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas. Aman dijalankan berulang.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

## 1. Pengaturan

In [ ]:
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    roc_auc_score, average_precision_score, log_loss
)
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

UJI_CEPAT = False          # True = uji cepat, hasil BUKAN untuk penelitian

SEEDS = [42]               # tambah mis. [42, 43, 44] untuk mengulang CV (waktu x jumlah seed)
N_OUTER = 5
N_INNER = 5
MODEL_SEED = 42

TRAIN_PATH = "data/final_split/train_final.csv"
OUTPUT_DIR = "hasil_eksperimen/15_eksperimen_utama"
PARAMS_A_PATH = "hasil_eksperimen/15a_tuning_skenario_a/params_skenario_A.json"

if UJI_CEPAT:
    N_OUTER = 3
    N_INNER = 3
    OUTPUT_DIR = "hasil_eksperimen/15_uji_cepat"
    PARAMS_A_PATH = "hasil_eksperimen/15a_uji_cepat/params_skenario_A.json"

CKPT_DIR = os.path.join(OUTPUT_DIR, "ckpt")
os.makedirs(CKPT_DIR, exist_ok=True)

assert "artifacts" not in OUTPUT_DIR and not OUTPUT_DIR.startswith("data"), "Folder output tidak boleh di artifacts/data"

print("Mode uji cepat :", UJI_CEPAT)
print("Seeds CV luar  :", SEEDS)
print("N_OUTER/N_INNER:", N_OUTER, "/", N_INNER)
print("Folder output  :", OUTPUT_DIR)

## 2. Hiperparameter Skenario A dan B

Skenario B memakai hasil tuning notebook 07 dan 08 (sama dengan yang dipakai notebook 10 sampai 13). Skenario A memakai hasil notebook 15a. Jika file hasil 15a belum ada, sel ini memakai **hiperparameter yang sama dengan B tetapi tanpa bobot**. Itu hanya pembanding sementara (efek bobot saja), bukan Skenario A sesuai proposal.

In [ ]:
RF_B = {"n_estimators": 400, "min_samples_split": 10, "min_samples_leaf": 2,
        "max_features": "sqrt", "max_depth": 20, "bootstrap": False}
XGB_B = {"n_estimators": 300, "min_child_weight": 5, "max_depth": 4,
         "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 1.0}

if os.path.exists(PARAMS_A_PATH):
    with open(PARAMS_A_PATH, encoding="utf-8") as f:
        _a = json.load(f)
    RF_A = dict(_a["rf"]["best_params"])
    XGB_A = dict(_a["xgb"]["best_params"])
    SUMBER_A = "hasil tuning sendiri (notebook 15a)"
else:
    RF_A = dict(RF_B)
    XGB_A = dict(XGB_B)
    SUMBER_A = "SEMENTARA: hiperparameter sama dengan B tanpa bobot (15a belum dijalankan)"

if UJI_CEPAT:
    for _p in (RF_A, RF_B):
        _p["n_estimators"] = 20
    for _p in (XGB_A, XGB_B):
        _p["n_estimators"] = 30

PARAMS = {"A": {"rf": RF_A, "xgb": XGB_A}, "B": {"rf": RF_B, "xgb": XGB_B}}

print("Sumber hiperparameter skenario A:", SUMBER_A)
for skenario in ["A", "B"]:
    print(f"\nSkenario {skenario}:")
    print("  RF :", PARAMS[skenario]["rf"])
    print("  XGB:", PARAMS[skenario]["xgb"])

## 3. Membaca Data Train Final

Hanya `train_final.csv`. Pada setiap lipatan juga dicatat apakah baris yang dinilai punya **kembaran persis** (semua 21 fitur sama) di data latih lipatan itu. Ini untuk mengukur seberapa besar duplikat memengaruhi hasil.

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
X = df_train.drop(columns=["Diabetes_012"]).reset_index(drop=True)
y = df_train["Diabetes_012"].astype(int).to_numpy()

if UJI_CEPAT:
    X, _, y, _ = train_test_split(X, y, train_size=6000, stratify=y, random_state=0)
    X = X.reset_index(drop=True)

HASH_BARIS = pd.util.hash_pandas_object(X, index=False).to_numpy()

print("Ukuran data train :", X.shape)
print("Distribusi kelas  :", np.bincount(y).tolist())
print("Baris dengan kembaran persis di seluruh train:",
      int(pd.Series(HASH_BARIS).duplicated(keep=False).sum()))

## 4. Fungsi Model dan Pelatihan

In [ ]:
def buat_rf(skenario):
    p = dict(PARAMS[skenario]["rf"])
    p.update(random_state=MODEL_SEED, n_jobs=-1)
    if skenario == "B":
        p["class_weight"] = "balanced"
    return RandomForestClassifier(**p)


def buat_xgb(skenario):
    p = dict(PARAMS[skenario]["xgb"])
    p.update(objective="multi:softprob", num_class=3, eval_metric="mlogloss",
             random_state=MODEL_SEED, n_jobs=-1)
    return XGBClassifier(**p)


def latih_rf(skenario, X_tr, y_tr):
    model = buat_rf(skenario)
    model.fit(X_tr, y_tr)
    return model


def latih_xgb(skenario, X_tr, y_tr):
    model = buat_xgb(skenario)
    if skenario == "B":
        model.fit(X_tr, y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
    else:
        model.fit(X_tr, y_tr)
    return model


def jalankan_satu(seed, tr, va, skenario):
    """Satu lipatan luar untuk satu skenario. Mengembalikan probabilitas validasi luar."""
    X_tr, y_tr = X.iloc[tr], y[tr]
    X_va, y_va = X.iloc[va], y[va]

    # Model dasar pada seluruh data latih luar -> RF/XGB tunggal sekaligus fitur validasi untuk Hybrid
    rf = latih_rf(skenario, X_tr, y_tr)
    p_rf = rf.predict_proba(X_va)
    del rf
    gc.collect()

    xg = latih_xgb(skenario, X_tr, y_tr)
    p_xgb = xg.predict_proba(X_va)
    del xg
    gc.collect()

    # OOF dalam: hanya dari data latih luar
    inner = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=seed)
    oof_rf = np.zeros((len(tr), 3))
    oof_xgb = np.zeros((len(tr), 3))
    for itr, iva in inner.split(X_tr, y_tr):
        m = latih_rf(skenario, X_tr.iloc[itr], y_tr[itr])
        oof_rf[iva] = m.predict_proba(X_tr.iloc[iva])
        del m
        m = latih_xgb(skenario, X_tr.iloc[itr], y_tr[itr])
        oof_xgb[iva] = m.predict_proba(X_tr.iloc[iva])
        del m
        gc.collect()

    z_tr = np.hstack([oof_rf, oof_xgb])
    z_va = np.hstack([p_rf, p_xgb])

    p_hyb = {}
    for kunci, bobot in [("hyb_nobal", None), ("hyb_bal", "balanced")]:
        meta = LogisticRegression(max_iter=2000, class_weight=bobot, random_state=42)
        meta.fit(z_tr, y_tr)
        p_hyb[kunci] = meta.predict_proba(z_va)

    twin = np.isin(HASH_BARIS[va], HASH_BARIS[tr])

    return {
        "idx": va, "y": y_va, "twin": twin,
        "rf": p_rf, "xgb": p_xgb,
        "hyb_nobal": p_hyb["hyb_nobal"], "hyb_bal": p_hyb["hyb_bal"],
        "z_tr": z_tr.astype(np.float32), "y_tr": y_tr,
    }


def path_ckpt(seed, fold, skenario):
    return os.path.join(CKPT_DIR, f"seed{seed}_fold{fold}_{skenario}.npz")

## 5. Menjalankan Eksperimen (Bagian Terberat)

Setiap lipatan yang selesai langsung disimpan. Jika sel ini terhenti (komputer mati, kernel restart), jalankan lagi dari awal: lipatan yang sudah selesai dilewati.

In [ ]:
daftar_tugas = []
for seed in SEEDS:
    outer = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(outer.split(X, y), start=1):
        for skenario in ["A", "B"]:
            daftar_tugas.append((seed, fold, skenario, tr, va))

total = len(daftar_tugas)
sisa = [t for t in daftar_tugas if not os.path.exists(path_ckpt(t[0], t[1], t[2]))]
print(f"Total tugas: {total} | sudah selesai: {total - len(sisa)} | akan dikerjakan: {len(sisa)}")

waktu_tugas = []
for seed, fold, skenario, tr, va in sisa:
    mulai = time.time()
    hasil_satu = jalankan_satu(seed, tr, va, skenario)
    np.savez_compressed(path_ckpt(seed, fold, skenario), **hasil_satu)
    durasi = time.time() - mulai
    waktu_tugas.append(durasi)
    sisa_tugas = len(sisa) - len(waktu_tugas)
    perkiraan = np.mean(waktu_tugas) * sisa_tugas / 3600
    print(f"[seed {seed}] lipatan {fold}/{N_OUTER} skenario {skenario} selesai dalam {durasi/60:.1f} menit | "
          f"sisa {sisa_tugas} tugas, perkiraan {perkiraan:.1f} jam")

print("\nSemua tugas selesai.")

## 6. Fungsi Metrik

Semua metrik dihitung dari probabilitas validasi luar. Prediksi kelas memakai `argmax` (tanpa threshold). Pengaruh threshold dibahas terpisah di bagian 9.

Catatan: pada Skenario B dan pada Hybrid berbobot, probabilitas sengaja bergeser ke kelas minoritas oleh pembobotan. **Log Loss dan Brier pada konfigurasi berbobot tidak sebanding dengan konfigurasi tanpa bobot.** Gunakan metrik berbasis keputusan dan peringkat (Recall, Presisi, F1, G-Mean, ROC-AUC, PR-AUC) untuk perbandingan.

In [ ]:
KONFIGURASI = [
    ("A", "RF", "rf"),
    ("A", "XGB", "xgb"),
    ("A", "Hybrid_meta_tanpa_bobot", "hyb_nobal"),
    ("A", "Hybrid_meta_berbobot", "hyb_bal"),
    ("B", "RF", "rf"),
    ("B", "XGB", "xgb"),
    ("B", "Hybrid_meta_tanpa_bobot", "hyb_nobal"),
    ("B", "Hybrid_meta_berbobot", "hyb_bal"),
]


def metrik_dari_proba(y_true, proba):
    proba = proba / proba.sum(axis=1, keepdims=True)
    pred = proba.argmax(axis=1)
    out = {}
    rec = recall_score(y_true, pred, labels=[0, 1, 2], average=None, zero_division=0)
    pre = precision_score(y_true, pred, labels=[0, 1, 2], average=None, zero_division=0)
    f1k = f1_score(y_true, pred, labels=[0, 1, 2], average=None, zero_division=0)
    for k in range(3):
        out[f"Recall_{k}"] = rec[k]
        out[f"Precision_{k}"] = pre[k]
        out[f"F1_{k}"] = f1k[k]
    out["Macro_F1"] = f1k.mean()
    out["G_Mean"] = float(np.prod(rec) ** (1 / 3))
    out["Accuracy"] = accuracy_score(y_true, pred)
    try:
        out["ROC_AUC_macro"] = roc_auc_score(y_true, proba, multi_class="ovr", average="macro", labels=[0, 1, 2])
        out["ROC_AUC_1"] = roc_auc_score((y_true == 1).astype(int), proba[:, 1])
        out["PR_AUC_1"] = average_precision_score((y_true == 1).astype(int), proba[:, 1])
        out["PR_AUC_2"] = average_precision_score((y_true == 2).astype(int), proba[:, 2])
        out["Log_Loss"] = log_loss(y_true, proba, labels=[0, 1, 2])
        onehot = np.eye(3)[y_true]
        out["Brier"] = float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))
    except ValueError:
        for kunci in ["ROC_AUC_macro", "ROC_AUC_1", "PR_AUC_1", "PR_AUC_2", "Log_Loss", "Brier"]:
            out[kunci] = np.nan
    return out


METRIK_UTAMA = ["Recall_0", "Recall_1", "Recall_2", "Precision_1", "F1_1", "Macro_F1", "G_Mean",
                "Accuracy", "ROC_AUC_macro", "ROC_AUC_1", "PR_AUC_1", "Log_Loss", "Brier"]

## 7. Memuat Hasil dan Menghitung Metrik per Lipatan

In [ ]:
baris = []
data_ckpt = {}        # (seed, fold, skenario) -> isi npz
lengkap = 0

for seed in SEEDS:
    for fold in range(1, N_OUTER + 1):
        for skenario in ["A", "B"]:
            p = path_ckpt(seed, fold, skenario)
            if not os.path.exists(p):
                print(f"PERINGATAN: belum ada hasil seed {seed} lipatan {fold} skenario {skenario}")
                continue
            d = np.load(p)
            data_ckpt[(seed, fold, skenario)] = {k: d[k] for k in ["idx", "y", "twin", "rf", "xgb", "hyb_nobal", "hyb_bal"]}
            lengkap += 1
            for (sk, nama, kunci) in KONFIGURASI:
                if sk != skenario:
                    continue
                proba = d[kunci]
                y_va = d["y"]
                twin = d["twin"]
                for subset, mask in [("semua", np.ones(len(y_va), dtype=bool)), ("tanpa_kembaran", ~twin)]:
                    m = metrik_dari_proba(y_va[mask], proba[mask])
                    m.update(Seed=seed, Lipatan=fold, Skenario=sk, Model=nama, Subset=subset, N_Baris=int(mask.sum()))
                    baris.append(m)

print(f"Lipatan lengkap: {lengkap} dari {len(SEEDS) * N_OUTER * 2}")

metrik_fold = pd.DataFrame(baris)
kolom_depan = ["Seed", "Lipatan", "Skenario", "Model", "Subset", "N_Baris"]
metrik_fold = metrik_fold[kolom_depan + [c for c in metrik_fold.columns if c not in kolom_depan]]
metrik_fold.to_csv(os.path.join(OUTPUT_DIR, "metrik_per_fold.csv"), index=False)
print("Baris metrik per lipatan:", len(metrik_fold))

## 8. Ringkasan Rata-rata dan Simpangan Baku

Satuan "lipatan" = satu pasangan (seed, lipatan luar). Simpangan baku dihitung antar lipatan (`ddof=1`).

In [ ]:
kunci_grup = ["Skenario", "Model", "Subset"]
kolom_metrik = [c for c in metrik_fold.columns if c not in kolom_depan]

mean_df = metrik_fold.groupby(kunci_grup)[kolom_metrik].mean()
sd_df = metrik_fold.groupby(kunci_grup)[kolom_metrik].std(ddof=1)
n_df = metrik_fold.groupby(kunci_grup).size().rename("N_Lipatan")

ringkasan = mean_df.add_suffix("_mean").join(sd_df.add_suffix("_sd")).join(n_df).reset_index()
ringkasan.to_csv(os.path.join(OUTPUT_DIR, "ringkasan_mean_sd.csv"), index=False)


def tampil(subset, metrik):
    t = mean_df.xs(subset, level="Subset")[metrik].round(4).astype(str) + " ± " + \
        sd_df.xs(subset, level="Subset")[metrik].round(4).astype(str)
    return t


print("=" * 100)
print("RINGKASAN (rata-rata ± simpangan baku antar lipatan), seluruh baris validasi, prediksi = argmax")
print("=" * 100)
print(tampil("semua", ["Recall_0", "Recall_1", "Recall_2", "Macro_F1", "G_Mean"]).to_string())
print()
print(tampil("semua", ["Precision_1", "F1_1", "PR_AUC_1", "ROC_AUC_macro", "Accuracy"]).to_string())
print()
print("Log Loss dan Brier (hanya sebanding antar konfigurasi dengan perlakuan bobot yang sama):")
print(tampil("semua", ["Log_Loss", "Brier"]).to_string())

# Grafik
urutan = [(s, m) for s, m, _ in KONFIGURASI]
etiket = [s + "\n" + m.replace("Hybrid_meta_", "Hybrid\n") for s, m in urutan]
warna = ["#5B7DB1" if s == "A" else "#E08E45" for s, _ in urutan]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
for ax, metrik in zip(axes, ["Macro_F1", "G_Mean", "Recall_1", "PR_AUC_1"]):
    nilai = [mean_df.loc[(s, m, "semua"), metrik] for s, m in urutan]
    galat = [np.nan_to_num(sd_df.loc[(s, m, "semua"), metrik]) for s, m in urutan]
    ax.bar(range(len(urutan)), nilai, yerr=galat, color=warna, capsize=3)
    ax.set_xticks(range(len(urutan)))
    ax.set_xticklabels(etiket, fontsize=7)
    ax.set_title(metrik)
fig.suptitle("Skenario A (biru) vs B (oranye); batang galat = simpangan baku antar lipatan", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "grafik_perbandingan.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Perbandingan Berpasangan

Untuk setiap pasangan konfigurasi, dihitung selisih metrik pada **lipatan yang sama**. Dua uji dilaporkan:

- **Uji t terkoreksi (Nadeau dan Bengio)**: koreksi varians untuk lipatan CV yang saling tumpang tindih. Rasio data uji terhadap data latih = 1/(`N_OUTER` - 1).
- **Wilcoxon signed-rank**: dengan hanya 5 lipatan, nilai p terkecil yang mungkin adalah 0,0625, jadi tidak akan pernah di bawah 0,05. Ia hanya pelengkap.

Jika `SEEDS` berisi lebih dari satu seed, lipatan dari seed berbeda tidak sepenuhnya independen. Anggap hasil sebagai pendekatan.

**Cara membaca:** selisih = konfigurasi pertama dikurangi kedua. Selisih positif berarti konfigurasi pertama lebih tinggi. Perhatikan besar selisih dan konsistensi tanda di semua lipatan, bukan hanya nilai p.

In [ ]:
PASANGAN = [
    ("B vs A | RF", ("B", "RF"), ("A", "RF")),
    ("B vs A | XGB", ("B", "XGB"), ("A", "XGB")),
    ("B vs A | Hybrid meta tanpa bobot", ("B", "Hybrid_meta_tanpa_bobot"), ("A", "Hybrid_meta_tanpa_bobot")),
    ("B: Hybrid meta tanpa bobot vs XGB", ("B", "Hybrid_meta_tanpa_bobot"), ("B", "XGB")),
    ("B: Hybrid meta tanpa bobot vs RF", ("B", "Hybrid_meta_tanpa_bobot"), ("B", "RF")),
    ("B: Hybrid meta berbobot vs XGB", ("B", "Hybrid_meta_berbobot"), ("B", "XGB")),
    ("B: Hybrid meta berbobot vs RF", ("B", "Hybrid_meta_berbobot"), ("B", "RF")),
    ("B: Hybrid meta berbobot vs tanpa bobot", ("B", "Hybrid_meta_berbobot"), ("B", "Hybrid_meta_tanpa_bobot")),
    ("A: Hybrid meta tanpa bobot vs XGB", ("A", "Hybrid_meta_tanpa_bobot"), ("A", "XGB")),
    ("A: Hybrid meta tanpa bobot vs RF", ("A", "Hybrid_meta_tanpa_bobot"), ("A", "RF")),
]
METRIK_UJI = ["Macro_F1", "G_Mean", "Recall_1", "PR_AUC_1", "ROC_AUC_macro"]


def uji_pasangan(selisih, n_outer):
    selisih = np.asarray(selisih, dtype=float)
    selisih = selisih[~np.isnan(selisih)]
    J = len(selisih)
    if J < 2:
        return dict(N=J, Rata_Selisih=np.nan, SD_Selisih=np.nan, t_terkoreksi=np.nan, p_t=np.nan,
                    p_wilcoxon=np.nan, Lipatan_Positif=np.nan)
    rata = selisih.mean()
    var = selisih.var(ddof=1)
    if var == 0:
        t_stat, p_t = (np.inf if rata != 0 else 0.0), (0.0 if rata != 0 else 1.0)
    else:
        t_stat = rata / np.sqrt((1 / J + 1 / (n_outer - 1)) * var)
        p_t = 2 * stats.t.sf(abs(t_stat), df=J - 1)
    try:
        p_w = stats.wilcoxon(selisih).pvalue
    except ValueError:
        p_w = np.nan
    return dict(N=J, Rata_Selisih=rata, SD_Selisih=np.sqrt(var), t_terkoreksi=t_stat, p_t=p_t,
                p_wilcoxon=p_w, Lipatan_Positif=int((selisih > 0).sum()))


semua = metrik_fold[metrik_fold["Subset"] == "semua"]
hasil_uji = []
for nama_pasangan, (s1, m1), (s2, m2) in PASANGAN:
    a = semua[(semua.Skenario == s1) & (semua.Model == m1)].set_index(["Seed", "Lipatan"])
    b = semua[(semua.Skenario == s2) & (semua.Model == m2)].set_index(["Seed", "Lipatan"])
    gabung = a.join(b, lsuffix="_1", rsuffix="_2", how="inner")
    for metrik in METRIK_UJI:
        selisih = gabung[f"{metrik}_1"] - gabung[f"{metrik}_2"]
        r = uji_pasangan(selisih.values, N_OUTER)
        r.update(Pasangan=nama_pasangan, Metrik=metrik)
        hasil_uji.append(r)

tabel_uji = pd.DataFrame(hasil_uji)[["Pasangan", "Metrik", "N", "Rata_Selisih", "SD_Selisih", "Lipatan_Positif",
                                      "t_terkoreksi", "p_t", "p_wilcoxon"]]
tabel_uji.to_csv(os.path.join(OUTPUT_DIR, "uji_berpasangan.csv"), index=False)

tampil_uji = tabel_uji.copy()
for kolom in ["Rata_Selisih", "SD_Selisih", "t_terkoreksi"]:
    tampil_uji[kolom] = tampil_uji[kolom].map(lambda v: f"{v:+.4f}" if np.isfinite(v) else "-")
for kolom in ["p_t", "p_wilcoxon"]:
    tampil_uji[kolom] = tampil_uji[kolom].map(lambda v: f"{v:.3f}" if np.isfinite(v) else "-")
print(tampil_uji.to_string(index=False))

## 10. Pengaruh Threshold Prediabetes (Dioptimalkan pada OOF Gabungan)

Skenario A tanpa bobot hampir tidak pernah memprediksi Prediabetes dengan `argmax`, sehingga perbandingan dengan `argmax` saja bisa tidak adil. Bagian ini bertanya: **jika threshold Prediabetes sama-sama dioptimalkan untuk semua konfigurasi, apakah CSL masih memberi keuntungan?**

Aturan keputusan sama dengan `predict.py`: jika `P(kelas 1) >= threshold` maka kelas 1, jika tidak pilih antara kelas 0 dan 2 menurut probabilitas terbesar. Pilihan threshold: G-Mean tertinggi, lalu Recall Prediabetes, lalu Macro F1 (sama dengan notebook 12).

**Peringatan:** threshold dicari pada probabilitas gabungan yang dinilai juga, sehingga angka ini optimistis. Perlakuannya sama untuk semua konfigurasi, jadi cocok untuk membandingkan, bukan sebagai estimasi akhir.

In [ ]:
def metrik_dari_cm(cm):
    tp = np.diag(cm).astype(float)
    benar_per_kelas = cm.sum(axis=1).astype(float)
    pred_per_kelas = cm.sum(axis=0).astype(float)
    rec = np.divide(tp, benar_per_kelas, out=np.zeros(3), where=benar_per_kelas > 0)
    pre = np.divide(tp, pred_per_kelas, out=np.zeros(3), where=pred_per_kelas > 0)
    f1 = np.divide(2 * pre * rec, pre + rec, out=np.zeros(3), where=(pre + rec) > 0)
    return rec, pre, f1, float(np.prod(rec) ** (1 / 3)), float(f1.mean()), float(tp.sum() / cm.sum())


GRID_THRESHOLD = np.unique(np.concatenate([np.arange(0.002, 0.05, 0.002), np.arange(0.05, 0.96, 0.01)]))

baris_thr = []
for seed in SEEDS:
    for (skenario, nama, kunci) in KONFIGURASI:
        bagian = [data_ckpt[(seed, f, skenario)] for f in range(1, N_OUTER + 1) if (seed, f, skenario) in data_ckpt]
        if len(bagian) < N_OUTER:
            continue
        y_gab = np.concatenate([b["y"] for b in bagian])
        p_gab = np.vstack([b[kunci] for b in bagian])
        p_gab = p_gab / p_gab.sum(axis=1, keepdims=True)
        arg02 = np.where(p_gab[:, 0] >= p_gab[:, 2], 0, 2)

        terbaik, skor_terbaik, cm_terbaik = None, (-1, -1, -1), None
        for thr in GRID_THRESHOLD:
            pred = np.where(p_gab[:, 1] >= thr, 1, arg02)
            cm = np.bincount(y_gab * 3 + pred, minlength=9).reshape(3, 3)
            rec, pre, f1, gm, mf1, acc = metrik_dari_cm(cm)
            skor = (gm, rec[1], mf1)
            if skor > skor_terbaik:
                terbaik, skor_terbaik, cm_terbaik = thr, skor, cm

        rec, pre, f1, gm, mf1, acc = metrik_dari_cm(cm_terbaik)
        baris_thr.append({
            "Seed": seed, "Skenario": skenario, "Model": nama, "Threshold": round(float(terbaik), 3),
            "Recall_0": rec[0], "Recall_1": rec[1], "Recall_2": rec[2],
            "Precision_1": pre[1], "F1_1": f1[1], "Macro_F1": mf1, "G_Mean": gm, "Accuracy": acc,
        })

tabel_thr = pd.DataFrame(baris_thr)
tabel_thr.to_csv(os.path.join(OUTPUT_DIR, "threshold_oof_gabungan.csv"), index=False)
if len(tabel_thr):
    print(tabel_thr.round(4).to_string(index=False))
else:
    print("Belum ada seed dengan seluruh lipatan lengkap.")

## 11. Sensitivitas terhadap Baris Kembar (Duplikat)

Setiap baris validasi ditandai punya kembaran persis di data latih lipatannya atau tidak. Tabel ini membandingkan hasil pada **seluruh baris** dengan hasil **hanya baris tanpa kembaran**. Jika selisihnya kecil, keputusan mempertahankan duplikat tidak banyak memengaruhi kesimpulan.

In [ ]:
kolom_sens = ["Macro_F1", "G_Mean", "Recall_0", "Recall_1", "Recall_2"]
mean_sub = metrik_fold.groupby(["Skenario", "Model", "Subset"])[kolom_sens].mean()

baris_sens = []
for (s, m, _) in KONFIGURASI:
    semua_ = mean_sub.loc[(s, m, "semua")]
    tanpa_ = mean_sub.loc[(s, m, "tanpa_kembaran")]
    baris = {"Skenario": s, "Model": m}
    for k in kolom_sens:
        baris[f"{k}_semua"] = semua_[k]
        baris[f"{k}_tanpa_kembaran"] = tanpa_[k]
        baris[f"{k}_selisih"] = tanpa_[k] - semua_[k]
    baris_sens.append(baris)

tabel_sens = pd.DataFrame(baris_sens)
tabel_sens.to_csv(os.path.join(OUTPUT_DIR, "sensitivitas_kembaran.csv"), index=False)

n_tanpa = metrik_fold[(metrik_fold.Subset == "tanpa_kembaran")].groupby(["Skenario", "Model"])["N_Baris"].mean()
rata_kembaran = 1 - (n_tanpa / metrik_fold[(metrik_fold.Subset == "semua")].groupby(["Skenario", "Model"])["N_Baris"].mean())
print(f"Rata-rata persentase baris validasi yang punya kembaran di data latih: {rata_kembaran.mean()*100:.2f}%")
print()
print(tabel_sens[["Skenario", "Model", "Macro_F1_semua", "Macro_F1_tanpa_kembaran", "Macro_F1_selisih",
                  "G_Mean_semua", "G_Mean_tanpa_kembaran", "G_Mean_selisih"]].round(4).to_string(index=False))

## 12. Catatan Akhir

In [ ]:
print("Hasil tersimpan di:", OUTPUT_DIR)
for nama in sorted(os.listdir(OUTPUT_DIR)):
    print("  -", nama)

print("\nCATATAN:")
print("- test_final_locked.csv TIDAK digunakan. Folder artifacts dan data TIDAK diubah.")
print("- Estimasi CV ini sedikit optimistis karena hiperparameter dipilih pada data latih yang sama.")
if UJI_CEPAT:
    print("- MODE UJI CEPAT: hasil ini BUKAN untuk penelitian.")
if SUMBER_A.startswith("SEMENTARA"):
    print("- PERHATIAN: skenario A masih memakai hiperparameter sementara. Jalankan notebook 15a, lalu hapus folder ckpt skenario A")
    print("  (file seed*_fold*_A.npz) dan jalankan ulang notebook ini.")

## 13. Cara Membaca Hasil untuk Laporan

1. **Apakah CSL (B) membantu?** Lihat `B vs A` pada `uji_berpasangan.csv` untuk Macro-F1, G-Mean, Recall Prediabetes, dan PR-AUC. Lihat juga `threshold_oof_gabungan.csv`: jika setelah threshold dioptimalkan selisih A dan B mengecil, sebagian manfaat CSL hanyalah menggeser ambang keputusan.
2. **Apakah Hybrid lebih baik dari model tunggal?** Lihat pasangan `Hybrid vs XGB` dan `Hybrid vs RF`. Jika selisihnya kecil dibanding simpangan baku, jangan menyebut Hybrid "lebih baik", cukup "setara" dan jelaskan manfaat lain jika ada.
3. **Meta-learner berbobot atau tidak?** Lihat pasangan `Hybrid meta berbobot vs tanpa bobot`. Ini menjawab perbedaan antara proposal (tanpa bobot) dan kode (berbobot).
4. **Duplikat:** gunakan `sensitivitas_kembaran.csv` untuk menjelaskan keputusan mempertahankan atau menghapus duplikat.
5. Tulis keterbatasan: hiperparameter dipilih pada data latih yang sama, dan prediabetes secara umum sulit dipisahkan dengan 21 fitur survei.